# DNAscape notebook smoke checks

A small, reproducible verification of the public examples: array shapes, region resolution, deterministic forward fields, stochastic outputs, file round trips, formulas and masks. These checks complement the repository's test suite; they do not replace numerical model validation.

Run **Restart Kernel and Run All** with DNAscape installed. No external files or downloads are required. Temporary exports are written to `examples/outputs/smoke_checks/`.


In [1]:
from pathlib import Path
import os
import sys

# Works with Jupyter started in the repository root or in examples/.
PROJECT_DIR = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "dnascape" / "__init__.py").is_file()
    ),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Start Jupyter inside your DNAscape checkout.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
OUTPUT_DIR = (
    Path(
        os.environ.get("DNASCAPE_NOTEBOOK_OUTPUT", PROJECT_DIR / "examples" / "outputs")
    )
    / "smoke_checks"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dnascape import rmap, rsim, rload, rsave, select_profile_region
from dnascape.workflows import DataFilterSettings, evaluate_formula, apply_data_filter

plt.rcParams.update(
    {
        "figure.figsize": (8, 3.2),
        "figure.dpi": 100,
        "axes.grid": False,
        "font.size": 10,
        "axes.prop_cycle": plt.cycler(color=["#b03412", "#3498b8", "#68615b"]),
    }
)
print(f"Outputs: {OUTPUT_DIR}")


Outputs: C:\Users\Francisco Berkemeier\Dropbox\Main Work\PostDoc\My Papers\DNAscape\github\DNAscape-devmode\examples\outputs\smoke_checks


## 1. Region and resolution invariants

Both coarser means and finer interpolation preserve the half-open genomic interval. The number of returned samples follows the output bin width.


In [2]:
source_profile = 10.0 + np.sin(np.arange(100) / 10.0)
for resolution, method, count in [(10.0, "mean", 4), (0.5, "linear", 80)]:
    values, positions = select_profile_region(
        source_profile,
        raw_resolution_kb=1.0,
        interval_kb=(20.0, 60.0),
        output_resolution_kb=resolution,
        resample_method=method,
    )
    assert values.size == positions.size == count
    assert positions[0] == 20.0 and positions[-1] < 60.0
print("Region selection and resolution: passed")


Region selection and resolution: passed


## 2. Forward kinetics and inverse fitting

Check a small spatial model, replicated-fraction monotonicity and fork-density accounting. The time grid is deliberately long enough to capture completion.


In [3]:
dx = 2.0
x = np.arange(32) * dx
t = np.arange(0.0, 241.0, 5.0)
ir = 0.004 * (1.0 + 0.2 * np.sin(x / 10.0))
options = {
    "resolution_x": dx,
    "resolution_t": 5.0,
    "time_grid": t,
    "n_bins": x.size,
    "verbose": False,
}
fraction = rmap(
    {"initiation_rate": ir, "fork_speed": 1.4},
    "replicated_fraction_xt",
    assumptions=options,
)
assert fraction.shape == (x.size, t.size)
assert np.all((fraction >= 0) & (fraction <= 1))
assert np.all(np.diff(fraction, axis=1) >= -1e-9)
assert np.min(fraction[:, -1]) > 0.999
timing = np.trapezoid(1.0 - fraction, t, axis=1)
fit = rmap(
    {"replication_timing": timing, "fork_speed": 1.4},
    "initiation_rate",
    assumptions={
        "target_form": "space",
        "resolution_x": dx,
        "maxiter": 3,
        "refine_edge_kb": 0.0,
        "verbose": False,
    },
)
assert np.asarray(fit).shape == timing.shape and np.isfinite(fit).all()
forks = rmap(
    {"initiation_rate": ir, "fork_speed": 1.4}, "fork_densities_xt", assumptions=options
)
np.testing.assert_allclose(
    forks["fork_density_xt"],
    forks["right_fork_density_xt"] + forks["left_fork_density_xt"],
)
print("Mapping shapes and fork-density accounting: passed")


Mapping shapes and fork-density accounting: passed


## 3. Simulation units, reproducibility and newer outputs

Convert initiation density to a per-bin rate before simulation. Confirm exact repeatability under the same seed, origin firing over time, completed replicon lengths and total space–time fork density.


In [4]:
settings = {
    "resolution_x": dx,
    "sim_number": 12,
    "seed": 71,
    "verbose": False,
    "time_statsQ": True,
    "time_stats_xtQ": True,
    "time_grid": t,
    "max_rep_time": 600.0,
}
inputs = {"initiation_rate": ir * dx, "fork_speed": 1.4}
first = rsim(source=inputs, assumptions=settings)
second = rsim(source=inputs, assumptions=settings)
np.testing.assert_array_equal(first["replication_timing"], second["replication_timing"])
assert np.asarray(first["rep_times_per_sim"]).shape == (12, x.size)
assert np.isfinite(first["rep_times_per_sim"]).all()
assert np.asarray(first["origin_firing"]).shape == t.shape
assert np.asarray(first["replicon_lengths"]).size > 0
np.testing.assert_allclose(
    first["fork_density_xt"],
    first["right_fork_density_xt"] + first["left_fork_density_xt"],
)
np.testing.assert_allclose(
    np.sum(first["origin_firing"]), np.mean(first["num_origins"])
)
print("Simulation reproducibility and outputs: passed")


Simulation reproducibility and outputs: passed


## 4. Formula and mask invariants

Formulas preserve shape; region transforms preserve unselected samples and the source array. Inverting a mask changes which part of the region is edited.


In [5]:
original = ir.copy()
transformed = evaluate_formula("source * 2", {"source": ir})
np.testing.assert_array_equal(transformed, ir * 2)
settings = DataFilterSettings(
    enabled=True,
    selection="ranges",
    position_ranges="(10000,20000)",
    profile_units="bp",
    action="replace",
    value=0.0,
)
edited = apply_data_filter(ir, settings, positions=x)
mask = (x >= 10) & (x < 20)
assert np.all(edited[mask] == 0)
np.testing.assert_array_equal(edited[~mask], ir[~mask])
np.testing.assert_array_equal(ir, original)
from dataclasses import replace

inverted = apply_data_filter(ir, replace(settings, complement=True), positions=x)
assert np.all(inverted[~mask] == 0)
np.testing.assert_array_equal(inverted[mask], ir[mask])
print("Formulas, masks, inversion and source preservation: passed")


Formulas, masks, inversion and source preservation: passed


## 5. Supported export formats

Normalise a single loaded column to a vector before comparing values; text and table containers can preserve a column dimension.


In [6]:
for suffix in ("tsv", "txt", "csv", "npz"):
    path = OUTPUT_DIR / f"roundtrip_x.{suffix}"
    rsave(ir, path, data_form="space")
    restored = np.asarray(rload(path, data_form="space")).reshape(-1)
    np.testing.assert_allclose(restored, ir)
print("All DNAscape notebook smoke checks passed.")


All DNAscape notebook smoke checks passed.
